In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import loompy as lp

/home/jiafzhou/.conda/envs/pyscenic/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [4]:
from pyscenic.plotting import plot_binarization
from pyscenic.export import add_scenic_metadata

/home/jiafzhou/.conda/envs/pyscenic/lib/python3.10/site-packages/ctxcore/__init__.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [5]:
import pyscenic
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
adata_full = sc.read_h5ad("/transfer/hhu/adata_hvg_scANVI.h5ad")

In [8]:
adata_full

AnnData object with n_obs × n_vars = 1608439 × 5380
    obs: 'barcodes', 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_well', 'bc2_well', 'bc3_well', 'timepoint', 'RD.TOTL', 'RD.PASS', 'RD.UNIQ', 'N.SNP', 'BEST', 'SNG.1ST', 'SNG.LLK1', 'SNG.2ND', 'SNG.LLK2', 'SNG.LLK0', 'LLK1', 'LLK2', 'PRB.SNG1', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'dataset', 'n_counts', 'timepointPerDataset', 'drug', 'histoneTarget', 'histoneFunction', 'targetComplex', 'gexEffect', 'final_annotation_HH', '_scvi_batch', '_scvi_labels', 'celltype_scANVI', 'source', 'proba_ASC', 'proba_DTel_Neuron', 'proba_Glioblast', 'proba_IPC', 'proba_NE_1', 'proba_NE_2', 'proba_NE_3', 'proba_NonTel_Neuron', 'proba_

In [5]:
print("Timepoints:")
print(
    adata_full.obs["timepoint"]
    .value_counts(dropna=False)
    .to_string()
)

d30_mask = (
    adata_full.obs["timepoint"]
    .astype(str)
    .str.lower()
    .eq("d30")
)

print("\nD30 total cells:", d30_mask.sum())

print("\nDrugs at d30:")
print(
    adata_full.obs.loc[d30_mask, "drug"]
    .value_counts(dropna=False)
    .to_string()
)

Timepoints:
timepoint
d10     630284
d30     563699
d121    414456

D30 total cells: 563699

Drugs at d30:
drug
m124         26065
m127         25860
m121         25723
m122         24587
m126         24538
m128         24398
m120         23478
m138         23174
m130         22606
m114         21076
m108         20814
m125         20418
m119         19438
m113         19288
m133         19215
m135         18580
m112         18293
m129         17832
m116         17260
m123         16777
m110         16690
dmso         16240
m131         15967
m115         14581
m134         12994
m111         12980
m132         11263
m117          8323
m140          8232
short-117     4018
short-113     3849
m118          3196
short-112     3183
m109          2753
m137             6
m139             4


In [10]:
d30_composition = pd.crosstab(
    adata_full.obs.loc[d30_mask, "celltype_scANVI"],
    adata_full.obs.loc[d30_mask, "drug"],
    margins=True
)

print(d30_composition.to_string())

drug              dmso   m108  m109   m110   m111   m112   m113   m114   m115   m116  m117  m118   m119   m120   m121   m122   m123   m124   m125   m126   m127   m128   m129   m130   m131   m132   m133   m134   m135  m137   m138  m139  m140  short-112  short-113  short-117     All
celltype_scANVI                                                                                                                                                                                                                                                                          
NE_1                18     18    30     17     38    121     18      7     18      4     3     1      8     62     47    126     96     18     66    122    119     41     92     72     92     11      1     11      6     0     67     0     1         70         14         90    1525
NE_3               119    391    25     36    264    182    106     12    136    791     7     0    150    174    332    297   1338     17     23    238  

In [9]:
adata_full.obs["celltype_scANVI"].value_counts()

celltype_scANVI
NE_1             274680
RG               174007
NE_3             162199
Others           147038
VTel_Neuron      143262
NonTel_RG        122541
DTel_Neuron      114995
Glioblast        107549
NonTel_Neuron     97254
NE_2              96942
Retina            74428
ASC               46861
IPC               46683
Name: count, dtype: int64

In [11]:
CELLTYPE_COL = "celltype_scANVI"
print(
    adata_full.obs[CELLTYPE_COL]
    .value_counts(dropna=False)
    .to_string()
)

celltype_scANVI
NE_1             274680
RG               174007
NE_3             162199
Others           147038
VTel_Neuron      143262
NonTel_RG        122541
DTel_Neuron      114995
Glioblast        107549
NonTel_Neuron     97254
NE_2              96942
Retina            74428
ASC               46861
IPC               46683


In [12]:
#save drug info
drug_counts_table = (
    adata_full.obs.loc[d30_mask, "drug"]
    .astype(str)
    .str.lower()
    .value_counts()
    .rename_axis("drug")
    .reset_index(name="n_cells")
)

drug_counts_table.to_csv(
    "/home/jiafzhou/GRN_org/config/d30_drug_counts.tsv",
    sep="\t",
    index=False
)

In [15]:
#drug by cell type info
drug_celltype_counts = pd.crosstab(
    adata_full.obs.loc[d30_mask, "drug"]
    .astype(str)
    .str.lower(),
    adata_full.obs.loc[d30_mask, CELLTYPE_COL]
)

drug_celltype_counts.head()
drug_celltype_counts.to_csv(
    "/home/jiafzhou/GRN_org/config/d30_drug_celltype_counts.tsv",
    sep="\t"
)

In [17]:
MIN_CELLS_PER_DRUG = 1000

In [19]:
adata_full.obs.loc[d30_mask, "drug"].value_counts() >= MIN_CELLS_PER_DRUG

drug
m124          True
m127          True
m121          True
m122          True
m126          True
m128          True
m120          True
m138          True
m130          True
m114          True
m108          True
m125          True
m119          True
m113          True
m133          True
m135          True
m112          True
m129          True
m116          True
m123          True
m110          True
dmso          True
m131          True
m115          True
m134          True
m111          True
m132          True
m117          True
m140          True
short-117     True
short-113     True
m118          True
short-112     True
m109          True
m137         False
m139         False
Name: count, dtype: bool

In [52]:
drug_counts = (
    adata_full.obs.loc[d30_mask, "drug"]
    .astype(str)
    .str.lower()
    .value_counts()
)

included_drugs = drug_counts.index[
    drug_counts >= MIN_CELLS_PER_DRUG
].tolist()

excluded_drugs = drug_counts.index[
    drug_counts < MIN_CELLS_PER_DRUG
].tolist()

pd.Series(included_drugs, name="drug").to_csv(
    "/home/jiafzhou/GRN_org/config/included_drugs.txt",
    index=False,
    header=False
)

pd.Series(excluded_drugs, name="drug").to_csv(
    "/home/jiafzhou/GRN_org/config/excluded_drugs.txt",
    index=False,
    header=False
)